# Stage 4: uncertainty tail guard + deterministic trellis

This notebook is self-contained. It mounts Google Drive, checks out the repository, installs the locked environment, copies the data, and creates missing Stage 2 and Stage 3 prerequisites automatically. With Stage 3 already present, Stage 4 itself should take roughly 2--8 minutes on a CPU runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
data_dir = Path('/content/rogii-data')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'

if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)

drive_data = drive_root / 'data'
assert (drive_data / 'train').is_dir(), f'Missing train directory: {drive_data / "train"}'
if not (data_dir / 'train').is_dir():
    shutil.copytree(drive_data, data_dir, dirs_exist_ok=True)
artifact_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
STAGE2_RUN_ID = 'stage2_pf_trend_blend_full_v001'
stage2_dir = artifact_dir / STAGE2_RUN_ID
if not (stage2_dir / 'metrics.json').is_file():
    assert not stage2_dir.exists() or not any(stage2_dir.iterdir()), f'Incomplete run: {stage2_dir}'
    subprocess.run([
        'uv', 'run', 'rogii-train',
        '--config', 'configs/experiment/pf_trend_blend.yaml',
        '--run-id', STAGE2_RUN_ID,
        '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir),
    ], cwd=repo_dir, check=True)

In [ ]:
STAGE3_RUN_ID = 'stage3_residual_hgb_full_v001'
stage3_dir = artifact_dir / STAGE3_RUN_ID
if not (stage3_dir / 'metrics.json').is_file():
    assert not stage3_dir.exists() or not any(stage3_dir.iterdir()), f'Incomplete run: {stage3_dir}'
    subprocess.run([
        'uv', 'run', 'rogii-residual',
        '--config', 'configs/experiment/stage3_residual_hgb.yaml',
        '--base-run', str(stage2_dir),
        '--run-id', STAGE3_RUN_ID,
        '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir),
    ], cwd=repo_dir, check=True)
stage3_metrics = json.loads((stage3_dir / 'metrics.json').read_text())
print('Stage 3 RMSE:', stage3_metrics['pooled_rmse'])

In [ ]:
SMOKE_RUN_ID = 'stage4_tail_path_smoke_v001'
smoke_dir = artifact_dir / SMOKE_RUN_ID
if not (smoke_dir / 'metrics.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-stage4',
        '--config', 'configs/experiment/stage4_tail_path.yaml',
        '--base-run', str(stage3_dir),
        '--run-id', SMOKE_RUN_ID,
        '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir),
        '--limit-wells', '8',
    ], cwd=repo_dir, check=True)
json.loads((smoke_dir / 'metrics.json').read_text())

In [ ]:
RUN_ID = 'stage4_tail_path_full_v001'
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'metrics.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-stage4',
        '--config', 'configs/experiment/stage4_tail_path.yaml',
        '--base-run', str(stage3_dir),
        '--run-id', RUN_ID,
        '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir),
    ], cwd=repo_dir, check=True)
stage4_metrics = json.loads((run_dir / 'metrics.json').read_text())
stage4_metrics

In [ ]:
{
    'stage3_rmse': stage3_metrics['pooled_rmse'],
    'stage4_rmse': stage4_metrics['pooled_rmse'],
    'rmse_delta': stage4_metrics['pooled_rmse'] - stage3_metrics['pooled_rmse'],
    'stage3_well_p90': stage3_metrics['well_rmse_p90'],
    'stage4_well_p90': stage4_metrics['well_rmse_p90'],
    'stage3_well_max': stage3_metrics['well_rmse_max'],
    'stage4_well_max': stage4_metrics['well_rmse_max'],
}

In [ ]:
comparison_dir = artifact_dir / 'stage4_vs_stage3_v001'
if not (comparison_dir / 'comparison.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-compare',
        '--baseline', str(stage3_dir),
        '--candidate', str(run_dir),
        '--output-dir', str(comparison_dir),
    ], cwd=repo_dir, check=True)
json.loads((comparison_dir / 'comparison.json').read_text())